In [2]:
import pandas as pd
import numpy as np
import os

In [3]:
# Step 1: Create dataframe df_target from 'raw' data
data_raw = {
    'cust_id': [1, 2, 3, 4, 5, 6, 7, 8, 9, 10],
    'rule_10': [1, 1, 1, 0, 0, 0, 1, 1, 0, 0],
    'rule_15': [0, 0, 0, 0, 0, 1, 1, 0, 0, 0],
    'rule_18': [0, 1, 0, 1, 0, 1, 0, 1, 0, 1],
    'rule_20': [1, 1, 1, 0, 0, 0, 0, 1, 0, 1]
}
df_target = pd.DataFrame(data_raw)
print(df_target.head(100))

   cust_id  rule_10  rule_15  rule_18  rule_20
0        1        1        0        0        1
1        2        1        0        1        1
2        3        1        0        0        1
3        4        0        0        1        0
4        5        0        0        0        0
5        6        0        1        1        0
6        7        1        1        0        0
7        8        1        0        1        1
8        9        0        0        0        0
9       10        0        0        1        1


In [15]:
# # Step 2: Create dataframe df_key from 'key' data
# data_key = {
#     'rule': ['rule_10', 'rule_15', 'rule_18', 'rule_20'],
#     'exclusion_reason': ['invalid ID', 'Foreign', 'No credit card', 'Wrong promo code']
# }
# df_key = pd.DataFrame(data_key)
# print(df_key.head(100))

In [17]:
# # --- 2. Create df_key from a dictionary ---
# # As requested, df_key is now created from a Python dictionary.
# key_data_dict = {
#     'rule': ['rule_10', 'rule_15', 'rule_18', 'rule_20'],
#     'exclusion_reason': ['invalid ID', 'Foreign', 'No credit card', 'Wrong promo code']
# }

# df_key = pd.DataFrame(key_data_dict)
# print(df_key.head(100))

      rule  exclusion_reason
0  rule_10        invalid ID
1  rule_15           Foreign
2  rule_18    No credit card
3  rule_20  Wrong promo code


In [4]:
# --- 2. Create df_key from the 'key' data ---
import io

key_data = """rule,exclusion_reason
rule_10,invalid ID
rule_15,Foreign
rule_18,No credit card
rule_20,Wrong promo code
"""

df_key = pd.read_csv(io.StringIO(key_data))
print(df_key.head(100))

      rule  exclusion_reason
0  rule_10        invalid ID
1  rule_15           Foreign
2  rule_18    No credit card
3  rule_20  Wrong promo code


In [26]:
# --- 3. Generate the summary table based on exclusion rules ---

# Get the list of rule columns in the specified order from df_key [[1]][doc_1].
rule_columns = df_key['rule'].tolist()

summary_data = []
waterfall_check_rules = []

In [27]:
# This loop calculates n_cust and waterfall for each exclusion rule [[1]][doc_1].
for rule in rule_columns:
    n_cust = df_target[rule].sum()
    
    waterfall_check_rules.append(rule)
    mask = (df_target[waterfall_check_rules] == 0).all(axis=1)
    waterfall = mask.sum()

    summary_data.append({
        'rule': rule,
        'n_cust': n_cust,
        'waterfall': waterfall
    })

df_summary = pd.DataFrame(summary_data)
df_result = pd.merge(df_key, df_summary, on='rule')
df_result = df_result[['rule', 'exclusion_reason', 'n_cust', 'waterfall']]

In [28]:
# --- 4. Insert the initial state row at the top of the result table ---

# Create a new DataFrame for the row to be inserted.
# The 'waterfall' value is the total number of customers from the raw data [[1]][doc_1].
initial_row = pd.DataFrame([{
    'rule': '-',
    'exclusion_reason': '(Whole base before exclusion)',
    'n_cust': 0,
    'waterfall': len(df_target)
}])

In [29]:
# Concatenate the new initial row with the previously generated result table.
df_final_result = pd.concat([initial_row, df_result], ignore_index=True)

In [30]:
# --- 5. Prepare data for final display ---

# Get the last value from the 'waterfall' column for the final sentence.
# This is done *before* formatting the column as a string.
# The final waterfall value for rule_20 is 2 [[2]][doc_2].
eligible_customers = df_final_result['waterfall'].iloc[-1]

# Apply comma formatting to the 'n_cust' and 'waterfall' columns for display.
# This converts the numbers in these columns into formatted strings.
df_final_result['n_cust'] = df_final_result['n_cust'].apply(lambda x: f'{x:,}')
df_final_result['waterfall'] = df_final_result['waterfall'].apply(lambda x: f'{x:,}')


In [31]:
# --- Display the final result ---
print("--- Final Result ---")
print(df_final_result)
# print("\n" + "="*30 + "\n") # Separator for clarity
print("\n" + "="*30 ) # Separator for clarity

# --- Display the final sentence as requested ---

# Print the final sentence using the stored eligible_customers variable.
print(f"\nNo. of eligible customers = {eligible_customers:,}")

--- Final Result ---
      rule               exclusion_reason n_cust waterfall
0        -  (Whole base before exclusion)      0        10
1  rule_10                     invalid ID      5         5
2  rule_15                        Foreign      2         4
3  rule_18                 No credit card      5         2
4  rule_20               Wrong promo code      5         2


No. of eligible customers = 2


In [8]:
# # --- Display the final result ---
# print("--- df_target ---")
# print(df_target)
# print("\n--- df_key ---")
# print(df_key)
# print("\n--- Final Result ---")
# print(df_result)